In [1]:
import pandas as pd
import requests
import io

In [2]:
# Helper to convert Bengali numerals to English
def convert_bn_numerals(text):
    bn_to_en = {'০':'0', '১':'1', '২':'2', '৩':'3', '৪':'4', '৫':'5', '৬':'6', '৭':'7', '৮':'8', '৯':'9'}
    for bn, en in bn_to_en.items():
        text = text.replace(bn, en)
    return text

In [ ]:


all_pages = []
base_url = "https://erp.powergrid.gov.bd/w/generations/view_generations_bn?page="
page_no = 30

for page in range(1, page_no):
    url = f"{base_url}{page}"
    response = requests.get(url, verify=False)
    
    if response.status_code == 200:
        # Use pandas to read the HTML table directly
        tables = pd.read_html(io.StringIO(response.text))
        if tables:
            all_pages.append(tables[0])
            print(f"Page {page} extracted.")

# Combine all pages into one DataFrame
if all_pages:
    # 1. Combine all pages
    final_df = pd.concat(all_pages, ignore_index=True)
    
    # 2. FIX: Flatten MultiIndex columns if they exist
    if isinstance(final_df.columns, pd.MultiIndex):
        # Joins the levels with an underscore, e.g., ('ভারত', 'ত্রিপুরা') -> 'ভারত_ত্রিপুরা'
        final_df.columns = [
            '_'.join([str(c) for c in col if "Unnamed" not in str(c)]).strip() 
            for col in final_df.columns.values
        ]

    # 3. Convert all numerals to English
    # Note: Use .map() for pandas 2.1.0+, or .applymap() for older versions
    final_df = final_df.map(lambda x: convert_bn_numerals(str(x)) if pd.notnull(x) else x)
    
    

c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page 1 extracted.


c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page 2 extracted.
Page 3 extracted.


c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [12]:
column_map = {
    'তারিখ_তারিখ': 'Date',
    'সময়_সময়': 'Time',
    'উৎপাদন(মেঃওঃ)_উৎপাদন(মেঃওঃ)': 'Generation_MW',
    'চাহিদা(মেঃওঃ)_চাহিদা(মেঃওঃ)': 'Demand_MW',
    'লোডশেড_লোডশেড': 'Loadshed',
    'গ্যাস_গ্যাস': 'Gas',
    'তরল জ্বালানী_তরল জ্বালানী': 'LPG',
    'কয়লা_কয়লা': 'Coal',
    'হাইড্রো_হাইড্রো': 'Hydro',
    'সৌর_সৌর': 'Solar',
    'বায়ু_বায়ু': 'Wind',
    'ভারত_ভেড়ামারা এইচভিডিসি': 'India_Bheramara',
    'ভারত_ত্রিপুরা': 'India_Tripura',
    'ভারত_আদানি': 'India_Adani',
    'নেপাল_নেপাল': 'Nepal',
    'মন্তব্য_মন্তব্য': 'Comment'
    }
final_df.rename(columns=column_map, inplace=True)
   

In [13]:
final_df.columns

Index(['Date', 'Time', 'Generation_MW', 'Demand_MW', 'Loadshed', 'Gas', 'LPG',
       'Coal', 'Hydro', 'Solar', 'Wind', 'India_Bheramara', 'India_Tripura',
       'India_Adani', 'Nepal', 'Comment'],
      dtype='str')

In [14]:
cols = ['Generation_MW', 'Demand_MW', 'Loadshed', 'Gas', 'LPG',
       'Coal', 'Hydro', 'Solar', 'Wind', 'India_Bheramara', 'India_Tripura',
       'India_Adani', 'Nepal']
# 3. CLEANING: Remove hidden spaces first (Very Important!)
for col in cols:
    final_df[col] = final_df[col].astype(str).str.strip()

# 4. CONVERSION: Force text to NaN and numbers to Float/Int
final_df[cols] = final_df[cols].apply(pd.to_numeric, errors='coerce')

In [15]:
final_df.dtypes

Date                 str
Time                 str
Generation_MW      int64
Demand_MW          int64
Loadshed           int64
Gas                int64
LPG                int64
Coal               int64
Hydro              int64
Solar              int64
Wind               int64
India_Bheramara    int64
India_Tripura      int64
India_Adani        int64
Nepal              int64
Comment              str
dtype: object

In [17]:
# Extract Hour from Time column (HH:MM:SS)
# We ensure it handles nulls and non-standard strings
final_df['Hour'] = final_df['Time'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) and ':' in str(x) else None)

# Reorder columns to put 'Hour' after 'Time' for better readability
cols = list(final_df.columns)
if 'Hour' in cols and 'Time' in cols:
    time_idx = cols.index('Time')
    cols.insert(time_idx + 1, cols.pop(cols.index('Hour')))
    final_df = final_df[cols]
final_df

,Date,Time,Hour,Generation_MW,Demand_MW,Loadshed,Gas,LPG,Coal,Hydro,Solar,Wind,India_Bheramara,India_Tripura,India_Adani,Nepal,Comment
0,17-04-2026,07:00:00,7,12550,13050,478,4882,1193,3780,70,115,0,907,134,1469,0,NaN
1,17-04-2026,06:00:00,6,12611,13130,496,5000,1236,3779,70,10,0,907,140,1469,0,NaN
2,17-04-2026,05:00:00,5,12592,13150,533,5054,1116,3805,69,0,0,907,172,1469,0,NaN
3,17-04-2026,04:00:00,4,12531,13650,1069,5094,1042,3788,68,0,0,907,162,1470,0,NaN
4,17-04-2026,03:00:00,3,12634,13800,1114,5119,1102,3806,67,0,0,907,168,1465,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,11-04-2026,09:00:00,9,11211,11300,85,5316,563,2965,80,528,0,879,132,748,0,NaN
149,11-04-2026,08:00:00,8,10989,11060,68,5379,471,2978,80,322,0,879,126,754,0,NaN
150,11-04-2026,07:00:00,7,10616,10660,42,5506,279,2891,80,111,0,879,114,756,0,NaN
151,11-04-2026,06:00:00,6,9950,9950,0,5542,44,2550,80,6,0,880,118,730,0,NaN


In [18]:
# 4. Now save to Excel with index=False
final_df.to_excel("PowerGrid_Data_Fixed.xlsx", index=False)
print("Success! MultiIndex flattened and data saved.")

Success! MultiIndex flattened and data saved.
